In [58]:
import torch 
import os
import numpy as np
import torch.nn.functional as F

from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms

In [59]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [60]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'Data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf100_test_data = datasets.CIFAR100(root = 'Data', train = False, download= True, transform = transforms.ToTensor())

In [61]:
#Splitting Logic
train_size = 40000
val_size = 10000

# print(len(cf100_training_data))

cf10_train_subset, cf10_val_subset = random_split(cf10_training_data, [train_size,val_size])
#cf100_val_data = random_split(cf100_training_data, [len(cf100_training_data)*0.8, len(cf100_training_data)*0.2])

In [62]:
print(1)
training_loader = torch.utils.data.DataLoader(cf10_train_subset, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_subset, batch_size=32, shuffle=False)
#testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

1


In [63]:
class LesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X
        
model_2 = LesNet().to(device)

In [64]:
# model 2

class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        #add dropuot 
        #self.dropout = nn.Dropout2d(p = 0.3) # 30% chance to be 0

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 
        
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

model_2 = Model2().to(device)

In [65]:
# model 3

class Model3(nn.Module): #Batch Normalization
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.bn1 = nn.BatchNorm2d(6)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)
        self.bn2 = nn.BatchNorm2d(16)
        

        self.fc1 = nn.Linear(5*5*16, 120)
        self.bn3 = nn.BatchNorm1d(120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

      

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.bn1(self.conv1(X)))
        X = F.max_pool2d(X, (2,2))
        

        X = F.relu(self.bn2(self.conv2(X)))
        X = F.max_pool2d(X, (2,2))
       

        X = X.flatten(1)

        X = F.relu(self.bn3(self.fc1(X)))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

model_3 = Model3().to(device)

In [66]:
#Train for each epoch
def train_one_epoch(model, loader,optimizer, criterion):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, label in loader:
        if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += label.size(0)
        correct += predicted.eq(label).sum().item()

    average_loss = running_loss / len(loader)
    
    accuracy = correct/ total * 100
    return average_loss, accuracy

In [67]:
#validate each epoch
def validate(model, loader, criterion):
    model.eval() #Set model to eval mode 
    running_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, label in loader:
            if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, label)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += label.size(0)
            correct += predicted.eq(label).sum().item()

        average_loss = running_loss / len(loader)
        accuracy = correct/ total * 100
        return average_loss, accuracy

            

In [68]:
def run_full_training_validate(model, train_loader, val_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = {"training_loss": [], "training_accuracy": [], "validation_loss": [], "validation_accuracy": []}
    print("---Training and Validation has started---")
    for epoch in range(epochs):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch+1}, Training Loss : {train_loss}, Training Accuracy: {train_accuracy}")
        val_loss, val_accuracy = validate(model, val_loader, criterion)
        print(f"Epoch {epoch+1}, Validation Loss : {val_loss}, Validation Accuracy: {val_accuracy}")

        history['training_loss'].append(train_loss)
        history['training_accuracy'].append(train_accuracy)
        history['validation_loss'].append(val_loss)
        history['validation_accuracy'].append(val_accuracy)

    print('---Training and Validation has ended---')
    return history

In [69]:
history_1 = run_full_training_validate(model_1, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 0.47717649344205854, Training Accuracy: 83.845
Epoch 1, Validation Loss : 0.46113308400106123, Validation Accuracy: 83.89999999999999
Epoch 2, Training Loss : 0.4110220906794071, Training Accuracy: 85.5925
Epoch 2, Validation Loss : 0.5028687900247665, Validation Accuracy: 82.25
Epoch 3, Training Loss : 0.37910881481915715, Training Accuracy: 86.6575
Epoch 3, Validation Loss : 0.5706677296386359, Validation Accuracy: 80.69
Epoch 4, Training Loss : 0.36378168551921847, Training Accuracy: 87.01
Epoch 4, Validation Loss : 0.5995560029920298, Validation Accuracy: 79.63
Epoch 5, Training Loss : 0.3418550455093384, Training Accuracy: 87.655
Epoch 5, Validation Loss : 0.6763312746636784, Validation Accuracy: 77.60000000000001
Epoch 6, Training Loss : 0.32215221123099325, Training Accuracy: 88.465
Epoch 6, Validation Loss : 0.7194605900551945, Validation Accuracy: 76.88000000000001
Epoch 7, Training Loss : 0.3197973086953163, T

In [71]:
history_2 = run_full_training_validate(model_2, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 1.963563362979889, Training Accuracy: 27.487499999999997
Epoch 1, Validation Loss : 1.8669435539946388, Validation Accuracy: 32.46
Epoch 2, Training Loss : 1.7507014573097228, Training Accuracy: 36.1375
Epoch 2, Validation Loss : 1.7038251443411976, Validation Accuracy: 37.87
Epoch 3, Training Loss : 1.6751953455924988, Training Accuracy: 39.5075
Epoch 3, Validation Loss : 1.6433648967895265, Validation Accuracy: 39.72
Epoch 4, Training Loss : 1.627303578567505, Training Accuracy: 41.3325
Epoch 4, Validation Loss : 1.6062968588484743, Validation Accuracy: 42.41
Epoch 5, Training Loss : 1.589516996860504, Training Accuracy: 42.620000000000005
Epoch 5, Validation Loss : 1.6004253142177107, Validation Accuracy: 41.699999999999996
Epoch 6, Training Loss : 1.5584210875511169, Training Accuracy: 43.5725
Epoch 6, Validation Loss : 1.5696883651014335, Validation Accuracy: 43.169999999999995
Epoch 7, Training Loss : 1.5351951504

In [72]:
history_3 = run_full_training_validate(model_3, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 1.5286219340324403, Training Accuracy: 44.545
Epoch 1, Validation Loss : 1.3496324501860255, Validation Accuracy: 51.42
Epoch 2, Training Loss : 1.273998682165146, Training Accuracy: 54.63249999999999
Epoch 2, Validation Loss : 1.2275402014628767, Validation Accuracy: 56.230000000000004
Epoch 3, Training Loss : 1.1690119641304015, Training Accuracy: 58.605
Epoch 3, Validation Loss : 1.1849642428346336, Validation Accuracy: 57.36
Epoch 4, Training Loss : 1.0823900083065032, Training Accuracy: 61.695
Epoch 4, Validation Loss : 1.1451760210549107, Validation Accuracy: 58.89
Epoch 5, Training Loss : 1.0197761810302735, Training Accuracy: 63.7525
Epoch 5, Validation Loss : 1.1534523562120553, Validation Accuracy: 59.31999999999999
Epoch 6, Training Loss : 0.966273570394516, Training Accuracy: 65.745
Epoch 6, Validation Loss : 1.2459344248802136, Validation Accuracy: 56.35
Epoch 7, Training Loss : 0.9182774335861206, Training

In [70]:
# def training_model(model, nr_epochs):
#     criterion = nn.CrossEntropyLoss()
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#     losses = []
#     for epoch in range(nr_epochs):
#         running_loss = 0

#         for i , data in enumerate(training_loader):
#             inputs, label = data

#             if torch.cuda.is_available():
#                 inputs, label = inputs.cuda(), label.cuda()
#                 model.cuda()

#             else:
#                 model.cpu()

#             optimizer.zero_grad()
#             outputs = model.forward(inputs)
#             loss = criterion(outputs, label)

            

#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item()

           
#         print(f"Epoch {epoch+1}, Loss : {running_loss/len(training_loader)}")


            
# training_model(model_1, 20)